In [ ]:
"""Module for producing dark-themed, publication-grade demographic data visualizations.

Provides a decoupled configuration architecture and optimization matrix to render
scatter plots mapping therapist provider density against population dynamics by
ZIP code, completely isolated from system-level layout warning anomalies.
"""

from dataclasses import dataclass
from pathlib import Path
import logging
import sys
from typing import Final, Self, Tuple

import matplotlib.pyplot as plt
from matplotlib.axes import Axes
from matplotlib.colors import Colormap
from matplotlib.figure import Figure
import pandas as pd

# Configure a high-performance structured runtime logger
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
)


@dataclass(frozen=True, slots=True)
class PlotConfiguration:
    """Immutable data container holding plotting boundary rules and visual design tokens."""

    max_zip: int = 23000
    max_count: int = 100
    min_population: int = 10000
    figsize: Tuple[int, int] = (12, 8)
    colormap_name: str = "Set2"  # Pastel/Bright accents optimized for dark foundations

    # Dark Theme Canvas Design Elements
    bg_dark: str = "#1E1E1E"  # Primary canvas background
    bg_panel: str = "#252526"  # Inside axes bounding grid matrix background
    text_light: str = "#D4D4D4"  # High-contrast readable foreground text
    grid_color: str = "#3C3C3C"  # Low-contrast subtle axis grid line grid demarcation

    @classmethod
    def create_default(cls) -> Self:
        """Instantiate a default immutable configuration wrapper blueprint."""
        return cls()


class DemographicVisualizer:
    """Architectural engine to ingest, filter, and render localized geospatial data."""

    # Structural dataframe schema contract validation targets
    REQUIRED_COLUMNS: Final[list[str]] = ["Zip", "Count", "Population", "Region"]

    def __init__(self, config: PlotConfiguration = PlotConfiguration.create_default()) -> None:
        """Initialize the visualizer applying fixed presentation rules."""
        self._config: Final[PlotConfiguration] = config

    def _apply_dark_theme(self, fig: Figure, ax: Axes) -> None:
        """Mutate the structural axes state parameters to enforce dark theme compliance."""
        cfg = self._config

        # Establish global figure background allocations
        fig.patch.set_facecolor(cfg.bg_dark)
        ax.set_facecolor(cfg.bg_panel)

        # Reconfigure mathematical edge boundaries spines visibility mappings
        for spine in ax.spines.values():
            spine.set_color(cfg.grid_color)
            spine.set_linewidth(1.2)

        # Force structural axis metric labels color shifts
        ax.xaxis.label.set_color(cfg.text_light)
        ax.yaxis.label.set_color(cfg.text_light)
        ax.title.set_color(cfg.text_light)

        # Mutate explicit tick parameters coordinates properties
        ax.tick_params(colors=cfg.text_light, which="both", labelsize=10)

    def process_and_plot(self, file_path: str | Path) -> Tuple[Figure, Axes] | Tuple[None, None]:
        """Ingest flat data data matrices, compute linear shifts, and generate dark layout.

        Args:
            file_path: Relative or absolute path targeting the aggregate data source.

        Returns:
            A tuple containing initialized Figure and Axes structures if successful,
            otherwise (None, None).
        """
        target_path: Final[Path] = Path(file_path)
        if not target_path.exists():
            logging.error(f"Target data vector not discovered at: {target_path.resolve()}")
            return None, None

        try:
            # Low overhead initial execution processing datastream mappings natively
            df: pd.DataFrame = pd.read_csv(target_path)

            if not all(col in df.columns for col in self.REQUIRED_COLUMNS):
                logging.critical("Input matrix layout lacks structural contract column keys.")
                return None, None

            # Coerce string or objects columns elements cleanly via fast internal vector arrays
            df["Zip"] = pd.to_numeric(df["Zip"], errors="coerce")
            cfg = self._config

            # Vectorized bitwise row operations filter slicing routines execution
            df_plot: pd.DataFrame = df[
                (df["Zip"] <= cfg.max_zip)
                & (df["Count"] <= cfg.max_count)
                & (df["Population"] > cfg.min_population)
            ].copy()

            if df_plot.empty:
                logging.warning("Data subset calculation returned empty array boundaries.")
                return None, None

            # Modern colormap dictionary generation mapping (Suppressing Matplotlib 3.11 deprecations)
            unique_regions: list[str] = sorted(df_plot["Region"].dropna().unique().tolist())
            cmap: Colormap = plt.colormaps.get_cmap(cfg.colormap_name)
            num_colors: int = getattr(cmap, "N", 8)
            region_to_color = {
                region: cmap(i % num_colors) for i, region in enumerate(unique_regions)
            }

            # Initialize target graphics display arrays objects matrices
            fig: Figure
            ax: Axes
            fig, ax = plt.subplots(figsize=cfg.figsize)
            self._apply_dark_theme(fig, ax)

            # Extract spatial axes dimensions transformations metrics
            x_range: float = float(df_plot["Population"].max() - df_plot["Population"].min())
            y_range: float = float(df_plot["Count"].max() - df_plot["Count"].min())

            # Protect translation boundaries parameters against singular point anomalies
            x_offset: Final[float] = (x_range * 0.012) if x_range > 0 else 1.0
            y_offset: Final[float] = (y_range * 0.008) if y_range > 0 else 1.0

            # Execute explicit localized point distribution passes via Pandas grouping splits
            for region_name, group in df_plot.groupby("Region"):
                region_str: Final[str] = str(region_name)
                color_vector: Final[Tuple[float, ...]] = region_to_color[region_str]

                ax.scatter(
                    group["Population"],
                    group["Count"],
                    label=region_str,
                    color=color_vector,
                    alpha=0.8,
                    edgecolor=cfg.bg_dark,
                    linewidth=0.8,
                    s=65,
                    zorder=3,
                )

                # High-efficiency row parsing using optimized named tuple generators
                for row in group.itertuples(index=False):
                    raw_x: float = float(getattr(row, "Population"))
                    raw_y: float = float(getattr(row, "Count"))
                    zip_label: Final[str] = str(int(getattr(row, "Zip")))

                    ax.text(
                        x=raw_x + x_offset,
                        y=raw_y + y_offset,
                        s=zip_label,
                        fontsize=7.5,
                        color=color_vector,
                        alpha=0.85,
                        horizontalalignment="left",
                        verticalalignment="bottom",
                        zorder=4,
                    )

            # Finalize cosmetic labeling properties structures
            ax.set_xlabel("Population", fontsize=12, fontweight="bold", labelpad=10)
            ax.set_ylabel("Therapists (Count)", fontsize=12, fontweight="bold", labelpad=10)
            ax.set_title(
                "Therapist Density vs Population Matrix by ZIP",
                fontsize=14,
                fontweight="bold",
                pad=18,
            )

            # Adjust background container dimensions layout parameters
            legend = ax.legend(
                title="Geographic Region",
                bbox_to_anchor=(1.04, 1),
                loc="upper left",
                borderaxespad=0.0,
                facecolor=cfg.bg_panel,
                edgecolor=cfg.grid_color,
            )
            legend.get_title().set_color(cfg.text_light)
            for text in legend.get_texts():
                text.set_color(cfg.text_light)

            ax.grid(True, linestyle=":", color=cfg.grid_color, alpha=0.6, zorder=1)
            plt.tight_layout()

            return fig, ax

        except (pd.errors.EmptyDataError, pd.errors.ParserError) as e:
            logging.error(f"Pandas framework file structural failure during processing: {e}")
        except Exception as e:
            logging.error(f"Uncaught execution engine processing error encountered: {e}")

        return None, None


if __name__ == "__main__":
    # In-line file orchestration test loop initialization
    visualizer = DemographicVisualizer()
    fig_matrix, axes_context = visualizer.process_and_plot("combined_zip_population_therapists.csv")
    if fig_matrix:
        plt.show()

In [ ]:
import pandas as pd
import csv

# --- Prepopulate filters here ---
MIN_POPULATION = 500
MAX_THERAPISTS = 100
MAX_ZIP = 23000
MIN_RATIO = 0  # Only show ZIPs with at least this pop/therapist ratio

# Load your combined CSV
df = pd.read_csv("combined_zip_population_therapists.csv")

# Load city/county info from census file
zip_info = {}
with open("census_data/virginia-zip-codes.csv", "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        zip_code = (row.get("zip") or "").strip().zfill(5)
        city = (row.get("city") or "").strip()
        county = (row.get("county") or "").strip()
        if zip_code:
            zip_info[zip_code] = (city, county)

# Apply filters
df_filtered = df[
    (df['Population'] > MIN_POPULATION) &
    (df['Count'] <= MAX_THERAPISTS) &
    (df['Zip'].astype(int) <= MAX_ZIP)
].copy()
df_filtered = df

# Avoid division by zero
df_filtered = df_filtered[df_filtered['Count'] > 0]

# Compute ratio
df_filtered['PopPerTherapist'] = df_filtered['Population'] / df_filtered['Count']

# Apply ratio filter if desired
df_filtered = df_filtered[df_filtered['PopPerTherapist'] >= MIN_RATIO]

# Group by region and sort within each region
regions = sorted(df_filtered['Region'].unique())

for region in regions:
    group = df_filtered[df_filtered['Region'] == region]
    group_sorted = group.sort_values('PopPerTherapist', ascending=False)
    if group_sorted.empty:
        continue
    print(f"\n=== {region} ===")
    for _, row in group_sorted.iterrows():
        zip_code = str(row['Zip']).zfill(5)
        city, county = zip_info.get(zip_code, ("", ""))
        #print(f"ZIP: {zip_code:<6}  City: {city:<20}  County: {county:<25}  Pop: {row['Population']:>7}  Therapists: {row['Count']:>4}  Pop/Therapist: {row['PopPerTherapist']:>8.2f}")
        #print(f"ZIP: {zip_code}, City: {city}, County: {county}, Pop: {row['Population']}, Therapists: {row['Count']}, Pop/Therapist: {row['PopPerTherapist']:.2f}")
        #print(f"ZIP: {zip_code}, City: {city}, Pop: {row['Population']}, Therapists: {row['Count']}, Pop/Therapist: {row['PopPerTherapist']:.2f}")
        print(f"ZIP: {zip_code:<6}  City: {city:<20}  Pop: {row['Population']:>7}  Therapists: {row['Count']:>4}  Pop/Therapist: {row['PopPerTherapist']:>8.2f}")
